In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

/Users/mohinikathrotiya/Desktop/Langchain/lcenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are a helpful AI Assiatant.'),
        ('human', '{input}'),
        MessagesPlaceholder(variable_name = 'history'),
    ]
)

chain = prompt | model

In [7]:
store = {}

In [8]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [12]:
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key= 'input',
    history_messages_key='history'
)

In [14]:
config1 = {'configurable': {'session_id' :'user'}}

In [15]:
r1 = with_history.invoke({'input': 'Hey, I am Alex, and I want to learn about Gen AI.'}, config= config1)

In [16]:
store['user'].messages

[HumanMessage(content='Hey, I am Alex, and I want to learn about Gen AI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hi Alex! It's great that you're interested in learning about Generative AI (Gen AI). Generative AI refers to a class of artificial intelligence models that can generate new content, such as text, images, music, and more, based on the data they have been trained on. Here are some key concepts and areas you might want to explore:\n\n### 1. **What is Generative AI?**\n   - Generative AI models learn patterns from existing data and can create new, similar data. For example, they can write stories, create artwork, or even generate code.\n\n### 2. **Types of Generative Models:**\n   - **Generative Adversarial Networks (GANs):** These consist of two neural networks (a generator and a discriminator) that work against each other to produce realistic data.\n   - **Variational Autoencoders (VAEs):** These models encode input data into a latent space and then 

In [17]:
store

{'user': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hey, I am Alex, and I want to learn about Gen AI.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Alex! It's great that you're interested in learning about Generative AI (Gen AI). Generative AI refers to a class of artificial intelligence models that can generate new content, such as text, images, music, and more, based on the data they have been trained on. Here are some key concepts and areas you might want to explore:\n\n### 1. **What is Generative AI?**\n   - Generative AI models learn patterns from existing data and can create new, similar data. For example, they can write stories, create artwork, or even generate code.\n\n### 2. **Types of Generative Models:**\n   - **Generative Adversarial Networks (GANs):** These consist of two neural networks (a generator and a discriminator) that work against each other to produce realistic data.\n   - **Variational Autoencoders (VAEs):** These models enc

In [19]:
config2 = {'configurable': {'session_id': 'user1'}}

r2 = with_history.invoke({'input': 'Hello, AI Assistant. I am here to learn Algebra concepts from you.'}, config = config2)

In [20]:
store

{'user': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hey, I am Alex, and I want to learn about Gen AI.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Alex! It's great that you're interested in learning about Generative AI (Gen AI). Generative AI refers to a class of artificial intelligence models that can generate new content, such as text, images, music, and more, based on the data they have been trained on. Here are some key concepts and areas you might want to explore:\n\n### 1. **What is Generative AI?**\n   - Generative AI models learn patterns from existing data and can create new, similar data. For example, they can write stories, create artwork, or even generate code.\n\n### 2. **Types of Generative Models:**\n   - **Generative Adversarial Networks (GANs):** These consist of two neural networks (a generator and a discriminator) that work against each other to produce realistic data.\n   - **Variational Autoencoders (VAEs):** These models enc

In [21]:
store['user1'].messages

[HumanMessage(content='Hello, AI Assistant. I am here to learn Algebra concepts from you.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! I’d be happy to help you learn Algebra. What specific concepts or topics in Algebra are you interested in? Here are some common topics we can cover:\n\n1. **Basic Operations** (addition, subtraction, multiplication, division)\n2. **Variables and Expressions**\n3. **Equations and Inequalities**\n4. **Functions and Graphs**\n5. **Polynomials**\n6. **Factoring**\n7. **Quadratic Equations**\n8. **Systems of Equations**\n9. **Exponents and Radicals**\n\nLet me know where you’d like to start or if you have a specific question!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 35, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'a

#### Multi-User Conversational Memory (LangChain)

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI

# 0) Define the model 
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 1) Prompt (System + History + User Input)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are a helpful assistant. Use the chat history to answer accurately'),
        (MessagesPlaceholder(variable_name='history')),
        ('human', '{input}')
    ]
)

chain = prompt | model

In [4]:
# 2) In-Memory store: session_id -> ChatMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [5]:
# 3) Wrapping the Chain with Message history

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history'
)

In [6]:
# 4) Helper function to chat as different users

def chat(session_id: str, text: str) -> str:
    config = {'configurable': {'session_id': session_id}}
    res = with_history.invoke({'input': text}, config=config)
    return res.content

In [7]:
# 5) Demo: multiple users interacting independently

# User A conversation
print("UserA:", chat("userA", "Hi, my name is Alex."))
print("UserA:", chat("userA", "What's my name?"))  # should remember Alex

print("-" * 80)

UserA: Hi Alex! How can I assist you today?
UserA: Your name is Alex. How can I help you today, Alex?
--------------------------------------------------------------------------------


In [8]:
store['userA'].messages

[HumanMessage(content='Hi, my name is Alex.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi Alex! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 31, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DBReQ3O5CcmBImecqAXUIPlkvaY82', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c7ccb-ff94-7123-a452-7a31108af5d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 10, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 H

In [9]:
# User B conversation (separate memory)
print("UserB:", chat("userB", "Hello, I am Rachel. My favorite food is pizza."))
print("UserB:", chat("userB", "What is my favorite food?"))  # should remember pizza

print("-" * 80)

UserB: Hi Rachel! Pizza is a delicious choice. Do you have a favorite topping or style of pizza?
UserB: Your favorite food is pizza!
--------------------------------------------------------------------------------


In [10]:
store['userB'].messages

[HumanMessage(content='Hello, I am Rachel. My favorite food is pizza.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi Rachel! Pizza is a delicious choice. Do you have a favorite topping or style of pizza?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 36, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_084a28d6e8', 'id': 'chatcmpl-DBRezf8NqRDholfiVBj31iALbXmoI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c7ccc-89d4-7ce2-aa2c-bdcab0e04259-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 20, 'total_tokens': 56, 'input_token_details': {'audio': 